# CSE 4310
# Programming Assignment 4 (PA4)
Alex Nguyen | 1002097417

# Eigenfaces

## Task A (30 points)
1. Load all the 25 images in the "/Eigenfaces/Train"
2. Display all the original faces in a 5X5 grid.
3. Find the mean face image. Perform PCA on the training faces.
4. Display the mean face.
5. Diplay all the eigenfaces in a 5X5 grid.

## Task B (30 points)

1. Select k = 2 eigenfaces (eigenvectors that correspond to the largest eigenvalues).
2. Reconstruct the training faces and display the reconstructed faces in a 5X5 grid.
3. Repeat the process for k = 5 and k = 15. For each k, reconstruct the training faces and display the reconstructed image in a 5X5 grid.


## Task C (40 points)
1. Load all the test images from "/Eigenfaces/Test"
2. Project each test image on the k = 2 eigenvectors and detect if it's a face. If it's a face, find it's closest training image. Use euclidean distance to calculate distance.
3. Display all the results in an M X 2 table, where M is total no of test images. Each row of the table displays two images. The image on the left is the test image. The image on the right is it's closest image in the eigenfaces space (Display the closest Training Image) . If an image is classified as a non-face, then the second column in the table should be blank (Just display a black image).
4. Repeat the process and display the results for k = 5, k = 15.

In [ ]:
# TASK A
import os


import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
# Load Images

# We will make a function for this since I obviously will later have to do this for both train and test.
def load_images(folder, size=None):
    files = sorted(f for f in os.listdir(folder) if f.lower().endswith(".jpg"))

    images  = []
    for f in files:
        img = cv2.imread(os.path.join(folder,f), cv2.IMREAD_GRAYSCALE)
        if size is None:
            size = (img.shape[1], img.shape[0])
        if (img.shape[1], img.shape[0]) != size:
            img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
        images.append(img.astype(np.float64) / 255.0)
    return np.stack(images), files

In [ ]:
# Find Mean face
# I intended to make this a function when I first planned skeleton code, but now I see getting mean face is easy.
def find_mean_face(images):
    return images.mean(axis=0)

In [ ]:
# PCA time
def perform_PCA(images):
    # Must reshape so we get less eigenfaces (25) 
    n = images.shape[0]
    X = images.reshape(n, -1)
    mean_face = find_mean_face(X) 

    # Compute covar matrix :D
    A = X - mean_face
    L = A @ A.T
    eigvals, V = np.linalg.eigh(L)
    
    # Find most dominant eigevector now
    order = np.argsort(eigvals)[::-1]
    eigals = eigvals[order]
    V = V[:, order]

    # I realized I can also compute eigenfaces in this step while I still have A
    U = A.T @ V
    U = U / np.linalg.norm(U, axis=0)
    eigenfaces = U.T

    return mean_face, eigenfaces

# After I makde this function get eigenfaces, it means the function is slightly misnamed, but that's OK for me

In [ ]:
# Helper to display images cleanly (lesson learned from doing manually everytime)
def show_grid(imgs, h, w, title, auto_contrast=False):
    fig, axes = plt.subplots(5, 5, figsize=(10, 10))
    for i, ax in enumerate(axes.ravel()):
        if auto_contrast:
            ax.imshow(imgs[i].reshape(h, w), cmap="gray")
        else:
            ax.imshow(imgs[i].reshape(h, w), cmap="gray", vmin=0, vmax=1)
        ax.set_title(str(i + 1), fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Task A
train_images, _ = load_images("./Eigenfaces/Train/")

n, h, w = train_images.shape
mean_face, eigenfaces = perform_PCA(train_images)

show_grid(train_images, h, w, "Original training faces")             # A.2
show_grid(eigenfaces, h, w, "Eigenfaces", auto_contrast=True)         # A.5

## Submission Guidelines:

    Submit through Canvas your source code in a single .ipynb file. The name of the .ipynb file should be YourStudentID.ipynb. (For example: 1001234567.ipynb)
    Import all the images from the ./Images directory. Your TA will use the same directory name to grade your submission.
    You don't need to attach the image file with your submission.